# Lezione 1 — Introduzione a Pandas per la Visualizzazione dei Dati

In questa lezione in diretta ripasseremo due idee fondamentali prima di scrivere un solo grafico:

1. **Perché visualizzare i dati?** — la teoria e l'intuizione.
2. **Domare i dati con Pandas** — il lavoro preliminare (preprocessing) che rende possibile qualsiasi visualizzazione utile.

Lavoreremo con un dataset reale: l'immigrazione verso il Canada tra il 1980 e il 2013 (fonte: Nazioni Unite).

## Parte 1 — Perché visualizzare i dati?

Immagina che ti diano una tabella con migliaia di righe e ti chiedano: "dimmi cosa sta succedendo qui". Guardando i numeri riga per riga ci metteresti ore e comunque ti sfuggirebbero i pattern. Il tuo cervello è progettato per riconoscere **forme, colori e tendenze** a colpo d'occhio, non per elaborare numeri isolati.

**Idea chiave:** una tabella ti dà i dati, un grafico ti racconta la storia che quei dati raccontano.

Prima di analizzare o modellare qualsiasi cosa, visualizzare ti aiuta a:
- Individuare errori (valori strani, buchi, duplicati).
- Capire la forma generale dei dati (sale? scende? ci sono gruppi?).
- Decidere quali domande ha senso porre ai dati.

### Domande diverse richiedono grafici diversi

Non esiste "il grafico giusto" in astratto, esiste il grafico giusto per la domanda che fai:

| Domanda | Tipo di grafico |
|---|---|
| Come è cambiato qualcosa nel tempo? | Linea |
| Come si confrontano le categorie? | Barre |
| Come si distribuisce un totale in parti? | Pie / treemap |
| Che relazione c'è tra due variabili? | Scatter (dispersione) |
| Dove succede qualcosa geograficamente? | Mappa |

Scegliere male il grafico non è solo una questione estetica: può far capire al pubblico qualcosa che i dati non dicono.

### La cassetta degli attrezzi del corso

| Libreria | A cosa serve? |
|---|---|
| **Matplotlib** | Base per creare grafici in Python. Molto controllo, devi costruire ogni dettaglio. |
| **Pandas** | La stessa libreria con cui carichi e pulisci i dati sa anche disegnare direttamente: `df.plot(kind='line')`. |
| **Seaborn** | Grafici statistici con estetica migliore di default. |
| **Plotly** | Visualizzazioni interattive (zoom, tooltip) — base delle dashboard con Dash. |
| **Folium** | Mappe interattive (marcatori, mappe coropletiche). |
| **PyWaffle** | Grafici "waffle" — alternativa visiva al pie per mostrare proporzioni. |

Non devi padroneggiarle tutte oggi: basta sapere che esistono e a cosa serve ciascuna.

### Punti chiave della Parte 1

- Una tabella e una visualizzazione non sono la stessa cosa: la tabella contiene i dati, la visualizzazione ti aiuta a capirli.
- La domanda che vuoi rispondere determina il tipo di grafico (tendenza → linea, confronto → barre, proporzione → pie, relazione → scatter, ubicazione → mappa).
- Con Pandas, un primo grafico può essere una sola riga: `df.plot(kind='line')`.

Ora sì, prima di creare qualsiasi grafico, andiamo a **domare i dati con Pandas**.

## Parte 2 — Domare i tuoi dati con Pandas prima di realizzare i grafici

Puoi conoscere tutti i trucchi di Matplotlib, ma se gli passi dati sporchi il grafico risultante è, nel migliore dei casi, inutile, e nel peggiore, fuorviante.

**Analogia:** i tuoi dati sono come ingredienti appena portati dal mercato — arrivano con la terra attaccata, foglie che non userai, non tagliati. Pandas è il tuo bancone da cucina: lì lavi (togli colonne inutili), tagli (selezioni ciò che interessa), organizzi in contenitori (indicizzi) e separi per tipo (filtri e raggruppi). Solo quando gli ingredienti sono pronti ha senso "cucinare" il grafico.

Faremo questo processo in diretta con il dataset dell'immigrazione verso il Canada (1980-2013).

### Installare le dipendenze

`pandas` ha bisogno del pacchetto `openpyxl` per leggere i file Excel (`.xlsx`). Su Google Colab lo installiamo con `pip`:

In [ ]:
!pip install openpyxl

### Caricare i dati

Importiamo `pandas` (gestione delle tabelle) e `numpy` (calcolo scientifico), e carichiamo il file Excel `Canada.xlsx` direttamente dal suo URL. Saltiamo le prime 20 righe (metadati inutili) e le ultime 2 righe (totali riassuntivi).

In [ ]:
import numpy as np   # utile per il calcolo scientifico
import pandas as pd  # la libreria principale per gestire dati in tabelle

df_can = pd.read_excel(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.xlsx',
    sheet_name='Canada by Citizenship',
    skiprows=range(20),   # saltiamo le prime 20 righe (non sono dati)
    skipfooter=2)         # ignoriamo le ultime 2 righe (totali riassuntivi)

print('Dati caricati in un DataFrame di pandas!')

### Esplorare prima di toccare qualsiasi cosa

Non iniziare mai a modificare un dataset senza prima guardarlo. `df_can.shape` dovrebbe essere la prima riga di codice che scrivi con qualsiasi nuovo dataset.

In [ ]:
# Vedi le prime 5 righe
df_can.head()

In [ ]:
# Vedi le ultime 5 righe
df_can.tail()

In [ ]:
# Riepilogo breve: quante colonne, tipi di dati, memoria usata
df_can.info(verbose=False)

In [ ]:
# Dimensioni: (numero di righe, numero di colonne)
df_can.shape

In [ ]:
# Statistiche descrittive delle colonne numeriche
df_can.describe()

### Pulire: rimuovere e rinominare colonne

Il dataset porta colonne (`AREA`, `REG`, `DEV`, `Type`, `Coverage`) che sono metadati della fonte e non aggiungono nulla alla nostra analisi. Le eliminiamo con `.drop()`, e poi rinominiamo le colonne con nomi poco chiari usando `.rename()`.

In [ ]:
# axis=0 sono le righe (default), axis=1 sono le colonne
df_can.drop(['AREA', 'REG', 'DEV', 'Type', 'Coverage'], axis=1, inplace=True)
df_can.head(2)

In [ ]:
df_can.rename(columns={'OdName': 'Country', 'AreaName': 'Continent', 'RegName': 'Region'}, inplace=True)
df_can.columns

### Creare una nuova colonna

Sommiamo tutti gli anni (dal 1980 al 2013) riga per riga per ottenere il totale di immigrati per paese.

In [ ]:
df_can['Total'] = df_can.sum(axis=1, numeric_only=True)
df_can['Total']

In [ ]:
# Controlliamo se ci sono valori nulli
df_can.isnull().sum()

### Indicizzare: `.set_index()`, `.loc` e `.iloc`

Di default l'indice è un intervallo numerico (0, 1, 2, ...), il che rende tedioso cercare un paese specifico. Trasformiamo la colonna `Country` nell'indice.

In [ ]:
df_can.set_index('Country', inplace=True)

# opzionale: rimuovere il nome "Country" dall'indice, è solo estetico
df_can.index.name = None

df_can.head(3)

Con il paese come indice abbiamo due modi per selezionare le righe:

- **`.loc[etichetta]`** seleziona in base al nome/etichetta dell'indice o della colonna.
- **`.iloc[posizione]`** seleziona in base alla posizione numerica.

`.loc` è più leggibile; `.iloc` è utile quando programmi in modo generico.

In [ ]:
# 1. L'intera riga del Giappone (tutte le colonne)
df_can.loc['Japan']

In [ ]:
# stesso risultato per posizione (il Giappone è nella posizione 87)
df_can.iloc[87]

In [ ]:
# 2. Solo l'anno 2013 per il Giappone
df_can.loc['Japan', 2013]

In [ ]:
# equivalente per posizione: 2013 è la colonna in posizione 36
df_can.iloc[87, 36]

In [ ]:
# 3. Un intervallo di anni per il Giappone
df_can.loc['Japan', [1980, 1981, 1982, 1983, 1984, 1985]]

Gli anni nelle colonne sono di tipo `int`. Per evitare ambiguità ("2013" è un anno o una posizione?), conviene convertire i nomi delle colonne in testo.

In [ ]:
df_can.columns = list(map(str, df_can.columns))

# variabile utile che useremo nei prossimi capitoli per creare i grafici
years = list(map(str, range(1980, 2014)))
years[:5]

### Filtrare righe con condizioni booleane

Per tenere solo le righe che soddisfano una condizione, passi un'espressione booleana tra parentesi quadre. Puoi combinare condizioni con `&` (e) e `|` (o) — mai con `and`/`or` del Python puro — e sempre tra parentesi.

In [ ]:
# 1. Creiamo la condizione (una serie di True/False, una per riga)
condition = df_can['Continent'] == 'Asia'

# 2. La passiamo al DataFrame per filtrare
df_can[condition].head()

In [ ]:
# Paesi dell'Asia che appartengono anche alla regione "Southern Asia"
df_can[(df_can['Continent'] == 'Asia') & (df_can['Region'] == 'Southern Asia')]

### Raggruppare con `groupby`

`groupby` risponde a domande del tipo "qual è il totale per categoria?".

In [ ]:
df_country = df_can.groupby(['Continent'])['Total'].sum().reset_index()
df_country

### Ordinare con `sort_values`

Per trovare i paesi che hanno contribuito di più al totale dell'immigrazione, ordiniamo per la colonna `Total`.

In [ ]:
df_can.sort_values(by='Total', ascending=False, axis=0, inplace=True)
top_5 = df_can.head(5)
top_5

### Punti chiave della Parte 2

- Nessun grafico sistema dati mal preparati: pulisci ed esplora prima, crea i grafici dopo.
- `.head()`, `.tail()`, `.info()`, `.shape`, `.describe()` sono il tuo primo contatto con qualsiasi nuovo dataset.
- `.drop()` elimina colonne/righe; `.rename()` dà loro nomi con senso.
- `.set_index()` facilita la ricerca per nome invece che per posizione numerica.
- `.loc` seleziona per etichetta; `.iloc` per posizione. Non sono intercambiabili.
- Filtrare significa passare una condizione booleana tra parentesi quadre: `df[condizione]`, combinando con `&`/`|` tra parentesi.
- `groupby()` raggruppa e aggrega; `sort_values()` ordina il risultato.

Nella pratica di oggi applicherai tutto questo per rispondere a domande concrete su questo stesso dataset.